In [1]:
import torch
import torch.nn as nn
from dotenv import load_dotenv
import os
from tqdm import tqdm

import warnings
warnings.filterwarnings("ignore")

In [2]:
import sys
from pathlib import Path

project_root = str(Path('.').resolve().parent)

if project_root not in sys.path:
    sys.path.append(project_root)

In [8]:
from model import build_transformer
from data import create_data_loader, train_tokenizers

from config import ConfigLoader

In [4]:
load_dotenv(dotenv_path="../.env", override=True)
hf_token = os.getenv("HF_TOKEN")

In [5]:
config = ConfigLoader.from_yaml("../config/parameters.yaml")

In [6]:
trained_tokens_dict = train_tokenizers(
    dataset_path=config["dataset_path"],
    dataset_name=config["dataset_name"],
    source_language=config["source_language"],
    target_language=config["target_language"],
    source_file_path=config["source_file_path"],
    target_file_path=config["target_file_path"],
    max_seq_len=config["max_seq_len"],
    hf_token=hf_token,
)

Filtered dataset: 997,064 / 1,000,000 pairs kept (99.7%)


In [7]:
trained_tokens_dict["source_seq_len"], trained_tokens_dict["target_seq_len"]

(128, 128)

In [8]:
train_data_loader = create_data_loader(
    dataset_path=config["dataset_path"],
    dataset_name=config["dataset_name"],
    split="train",
    trained_tokens_dict=trained_tokens_dict,
    source_language=config["source_language"],
    target_language=config["target_language"],
    batch_size=config["train_batch_size"],
    shuffle=True,
    hf_token=hf_token,
)

In [9]:
val_data_loader = create_data_loader(
    dataset_path=config["dataset_path"],
    dataset_name=config["dataset_name"],
    split="validation",
    trained_tokens_dict=trained_tokens_dict,
    source_language=config["source_language"],
    target_language=config["target_language"],
    batch_size=config["val_batch_size"],
    shuffle=True,
    hf_token=hf_token,
)

In [10]:
model = build_transformer(
    source_vocab_size=trained_tokens_dict["source_vocab_size"],
    target_vocab_size=trained_tokens_dict["target_vocab_size"],
    source_seq_len=trained_tokens_dict["source_seq_len"],
    target_seq_len=trained_tokens_dict["target_seq_len"],
    d_model=config["d_model"],
    h=config["num_heads"],
    dropout=config["dropout"],
    n_encoder=config["n_encoder"],
    n_decoder=config["n_decoder"],
)

In [11]:
data = next(iter(train_data_loader))

In [12]:
data["source_sentence"]

["Oh, for you, I'd practice morning, noon, and night.",
 "I don't know who your contacts are, morgan, but you are my up-and-comer.",
 'Anyone!',
 "I don't feel anything.",
 'Ball is also afraid of Yashin.',
 "I'm sorry about your father.",
 'Goddammit.',
 'Why, no.',
 'the same circuit of everidentical dwelling units , offices , freeways , .',
 "It's beautiful.",
 'Cheers.',
 'Thank you for coming in.']

In [13]:
data["encoder_input"].shape, data["decoder_input"].shape, data["label"].shape

(torch.Size([12, 128]), torch.Size([12, 128]), torch.Size([12, 128]))

In [14]:
data["encoder_input"]

tensor([[    0, 10483,    56,  ...,     2,     2,     2],
        [    0,  1785,  9555,  ...,     2,     2,     2],
        [    0,     3,    23,  ...,     2,     2,     2],
        ...,
        [    0,  9888,   747,  ...,     2,     2,     2],
        [    0,     3,     4,  ...,     2,     2,     2],
        [    0,     3,  2776,  ...,     2,     2,     2]])

In [15]:
data["encoder_mask"].shape, data["decoder_mask"].shape

(torch.Size([12, 1, 1, 128]), torch.Size([12, 1, 128, 128]))

In [16]:
data["encoder_mask"][0,0,0,:]

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.])

In [17]:
data["decoder_mask"][0,0,:5,:5]

tensor([[1., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0.],
        [1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1.]])

In [22]:
encoder_output = model.encode(data["encoder_input"], data["encoder_mask"])

In [23]:
encoder_output.shape

torch.Size([12, 128, 512])

In [24]:
decoder_output = model.decode(
    x=data["decoder_input"],
    encoder_output=encoded,
    encoder_mask=data["encoder_mask"],
    decoder_mask=data["decoder_mask"],
)

In [25]:
decoder_output.shape

torch.Size([12, 128, 512])

In [33]:
trained_tokens_dict.keys()

dict_keys(['source_tokenizer', 'target_tokenizer', 'source_seq_len', 'target_seq_len', 'source_vocab_size', 'target_vocab_size'])

In [2]:
import torch
import torch.nn as nn
from dotenv import load_dotenv
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

import sys
from pathlib import Path
project_root = str(Path('.').resolve().parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from model import build_transformer
from data import create_data_loader, train_tokenizers
from config import ConfigLoader

load_dotenv(dotenv_path="../.env", override=True)
hf_token = os.getenv("HF_TOKEN")





def prepare_model_data(
    config: dict,
    hf_token: str | None,
    
):
    trained_tokens_dict = train_tokenizers(
        dataset_path=config["dataset_path"],
        dataset_name=config["dataset_name"],
        source_language=config["source_language"],
        target_language=config["target_language"],
        source_file_path=config["source_file_path"],
        target_file_path=config["target_file_path"],
        max_seq_len=config["max_seq_len"],
        hf_token=hf_token,
    )

    train_data_loader = create_data_loader(
        dataset_path=config["dataset_path"],
        dataset_name=config["dataset_name"],
        split="train",
        trained_tokens_dict=trained_tokens_dict,
        source_language=config["source_language"],
        target_language=config["target_language"],
        batch_size=config["train_batch_size"],
        shuffle=True,
        hf_token=hf_token,
    )

    val_data_loader = create_data_loader(
        dataset_path=config["dataset_path"],
        dataset_name=config["dataset_name"],
        split="validation",
        trained_tokens_dict=trained_tokens_dict,
        source_language=config["source_language"],
        target_language=config["target_language"],
        batch_size=config["val_batch_size"],
        shuffle=True,
        hf_token=hf_token,
    )

    model = build_transformer(
        source_vocab_size=trained_tokens_dict["source_vocab_size"],
        target_vocab_size=trained_tokens_dict["target_vocab_size"],
        source_seq_len=trained_tokens_dict["source_seq_len"],
        target_seq_len=trained_tokens_dict["target_seq_len"],
        d_model=config["d_model"],
        h=config["num_heads"],
        dropout=config["dropout"],
        n_encoder=config["n_encoder"],
        n_decoder=config["n_decoder"],
    )

    return model, train_data_loader, val_data_loader, trained_tokens_dict


def train_model(
    config_file_path: str,
    hf_token: str,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)
    if (device == "cuda"):
        print(f"Device name: {torch.cuda.get_device_name(device.index)}")
        print(f"Device memory: {torch.cuda.get_device_properties(device.index).total_memory / 1024 ** 3} GB")
    torch.device(device)
    
    config = ConfigLoader.from_yaml(config_file_path)
    model, train_data_loader, val_data_loader, trained_tokens_dict = prepare_model_data(config=config, hf_token=hf_token)
    
    model = model.to(device)
    optimizer = torch.optim.AdamW(
        params=model.parameters(),
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )
    loss_fn = nn.CrossEntropyLoss(
        ignore_index=trained_tokens_dict["source_tokenizer"].token_to_id('<PAD>'),
        label_smoothing=0.1,
    )

    for epoch in range(config["num_epochs"]):
        batch_iterator = tqdm(train_data_loader, desc=f"Processing Epoch {epoch:02d}")
        model.train()
        for batch in batch_iterator:

            encoder_output = model.encode(
                batch["encoder_input"].to(device),
                batch["encoder_mask"].to(device),
            )
            decoder_output = model.decode(
                batch["decoder_input"].to(device),
                encoder_output,
                batch["encoder_mask"].to(device),
                batch["decoder_mask"].to(device),
            )
            projection = model.project(decoder_output)

            loss = loss_fn(projection.view(-1, trained_tokens_dict["target_vocab_size"]), batch["label"].to(device).view(-1))

            batch_iterator.set_postfix({"loss": f"{loss.item():6.3f}"})

            loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        torch.save(
            obj={
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            },
            f=f"otputs/model_outputs/model{epoch}.pt",
        )


train_model(
    config_file_path="../config/parameters.yaml",
    hf_token=hf_token,
)

Using device: cpu


Filter: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000000/1000000 [00:22<00:00, 43772.70 examples/s]


Filtered dataset: 997,064 / 1,000,000 pairs kept (99.7%)


Processing Epoch 00:   0%|                                                                                                                | 39/83089 [04:42<167:07:38,  7.24s/it, loss=4.430]


KeyboardInterrupt: 